In [14]:
import os

In [15]:
OPENAI_API_KEY="REMOVED_OPENAI_API_KEY"

In [16]:
os.environ["OPENAI_API_KEY"]=OPENAI_API_KEY

In [17]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4o-mini"
)

In [18]:
llm.invoke("Hello World!").content

'Hello! How can I assist you today?'

In [19]:
from flow import execute

In [20]:
chunks = execute()

[INFO] Starting RFP processing pipeline.
[INFO] Extracting text from PDF: /home/kparth/HomersHackers/parth/data/ELIGIBLE_RFP_2.pdf
[INFO] Extraction complete.
[INFO] Cleaning extracted text.
[INFO] Cleaning complete.
[INFO] Chunking the cleaned document.


In [21]:
chunks

['Hazelwood School District Request for Proposals RFP for Information Technology Audit Services Due : February 25 , 2025 Time : 10 : 00 a . m . CDT February 3 , 2025 REQUEST FOR PROPOSAL INFORMATION TECHNOLOGY AUDITING SERVICES The Hazelwood School District seeks proposals from experienced firms to conduct a comprehensive information technology audit for the Hazelwood School District HSD . This audit will collect and evaluate evidence of HSDs information technology systems , practices , and operations to determine if changes are needed in the existing structure to meet current and future needs . Any questions regarding the specifications are to be directed to Danielle Thomas , Director of Purchasing Supplier Diversity no later than 2 : 00 pm on Tuesday , February 18 , 2025 through the Vendor Registry online question submission process via the districts website at https : www . hazelwoodschools . orgPage2238 . Only these inquiries will be answered . Any items requiring clarification wil

In [22]:
print(len(chunks))

19


In [10]:
from langchain_core.prompts import ChatPromptTemplate

In [26]:
prompt_template = ChatPromptTemplate.from_template(
    """
You are provided with multiple chunks of content extracted from a Request for Proposal (RFP). Your task is to carefully analyze each chunk and identify the specific requirements that fall into the following category:

> **Non-Mandatory but Advantageous Requirements**  
> These are requirements or features that are not explicitly marked as mandatory or essential to qualify for the bid but are likely to enhance the bidder’s chances of winning the contract. These may be referred to as “preferred,” “desirable,” “optional,” “value-added,” or may be implied through phrases such as “it would be beneficial if…” or “vendors are encouraged to…”

**Your goal is to:**
1. Go through every provided RFP chunk carefully.
2. Extract and list the non-mandatory (but beneficial) requirements mentioned or implied in the text.
3. For each identified requirement, provide:
   - A brief summary of the requirement.
   - The exact language (quote) from the RFP that indicates it is non-mandatory.
   - A rationale for why this requirement could help increase the company’s chances of winning the bid.

**Make sure to:**
- Avoid listing mandatory requirements or eligibility criteria unless they are mentioned in contrast to the advantageous ones.
- Be meticulous and precise. Every relevant hint in the language should be considered.
- Present your output in a structured format for easy understanding.

---

**RFP Chunk to Analyze**:
{text}
"""
)

In [27]:
from langchain_core.runnables import Runnable
from langchain_core.output_parsers import StrOutputParser

In [28]:
# Create the chain
chain: Runnable = prompt_template | llm | StrOutputParser()

# Apply to chunks
results = [chain.invoke({"text": chunk}) for chunk in chunks]

In [ ]:
results

['Here is the analysis of the provided RFP chunk, focusing on non-mandatory but advantageous requirements:\n\n### Non-Mandatory but Advantageous Requirements\n\n#### Requirement 1: Demonstration of Relevant Experience\n- **Summary**: Vendors that can showcase previous successful audits or projects specifically within educational sectors or similar organizations may have an enhanced chance of winning the bid.\n- **Exact Language**: Though not explicitly mentioned in a direct quote from the RFP, the requirement can be inferred from the context: "The Hazelwood School District seeks proposals from experienced firms."\n- **Rationale**: Demonstrating relevant experience can build credibility and assure the school district of the vendor\'s capability to understand and meet the specific needs of an educational setting.\n\n#### Requirement 2: Value-Added Services\n- **Summary**: Proposals that include additional services or insights beyond the core audit requirements could stand out.\n- **Exact

In [11]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import Runnable
from langchain_openai import ChatOpenAI

### --- Phase 1: Filtering Prompt --- ###
filter_prompt = ChatPromptTemplate.from_template(
    """
You're given a chunk of an RFP document.

Your task is to determine whether this chunk contains **any** non-mandatory but advantageous requirements.

These are features or requests that are beneficial but not required — often using words like "preferred", "desirable", "value-added", "optional", or phrases like "it would be beneficial if", "vendors are encouraged to", etc.

Respond with only "Yes" or "No".

Chunk:
{text}
"""
)

# Create filtering chain
filter_chain: Runnable = filter_prompt | llm | StrOutputParser()

# Phase 1: Filter relevant chunks
relevant_chunks = []
for chunk in chunks:
    result = filter_chain.invoke({"text": chunk})
    if "yes" in result.lower():  # case-insensitive check
        relevant_chunks.append(chunk)


In [12]:
print(len(relevant_chunks))

0


In [13]:
relevant_chunks

[]

In [ ]:

### --- Phase 2: Deep Analysis Prompt --- ###
analysis_prompt = ChatPromptTemplate.from_template(
    """
You are provided with a chunk of content extracted from a Request for Proposal (RFP). Analyze the chunk below and identify the specific requirements that fall into the following category:

> **Non-Mandatory but Advantageous Requirements**  
> These are requirements or features that are not explicitly marked as mandatory or essential to qualify for the bid but are likely to enhance the bidder’s chances of winning the contract. These may be referred to as “preferred,” “desirable,” “optional,” “value-added,” or may be implied through phrases such as “it would be beneficial if…” or “vendors are encouraged to…”

**Your goal is to:**
1. Extract and list the non-mandatory (but beneficial) requirements mentioned or implied in the text.
2. For each, provide:
   - A brief summary of the requirement.
   - The exact language (quote) from the RFP that indicates it is non-mandatory.
   - A rationale for why this requirement could help increase the company’s chances of winning the bid.

**RFP Chunk**:
{text}
"""
)

# Build the analysis chain
analysis_chain: Runnable = analysis_prompt | llm | StrOutputParser()

# Phase 2: Analyze only the filtered chunks
final_results = [analysis_chain.invoke({"text": chunk}) for chunk in relevant_chunks]
